1. Import + Config

In [ ]:
# [CELL 1] IMPORT VÀ CẤU HÌNH THÔNG SỐ
import asyncio
import aiohttp
import pandas as pd
import json
import os
import base64
from pathlib import Path
from tqdm.asyncio import tqdm

# --- CONFIGURATION ---
API_KEY = "sk-824a080f44c7c8de2bd15ff4e13d386b4581ecc54ae1d506b049183d13a8c808"
API_URL = "https://ckey.vn/v1/chat/completions"
MODEL = "gpt-5.4-mini"

INPUT_CSV = "../data/processed/dataset_final_qwen_filled.csv"
OUTPUT_CSV = "../data/processed/dataset_metadata_filled.csv"
IMAGE_DIR = "../data/raw/images"
CHECKPOINT_FILE = "../data/processed/relabel_checkpoint.jsonl" 

BATCH_SIZE_TEXT = 20
BATCH_SIZE_VISION = 5 
CONCURRENCY_LIMIT = 15 

ENUMS = """
1. "style_aesthetic": [Minimalist/Basic, Streetwear/Y2K, Vintage/Retro, Bohemian/Resort, Sporty/Athleisure, Office/Smart Chic, Edgy/Grunge, Preppy/Academic, Glam/Party, Loungewear/Comfy, Intimate/Sensual, Unknown]
2. "dominant_material": [Cotton/Jersey, Denim/Chambray, Leather/Faux Leather, Linen/Hemp, Silk/Satin/Chiffon, Knitted/Ribbed, Wool/Fleece/Cashmere, Synthetic/Nylon/Polyester, Velvet/Corduroy, Lace/Mesh/Tulle, Unknown]
3. "design_detail": [Cropped, High-waisted, V-neck/Plunge, Turtleneck/High-neck, Sleeveless/Halter, Button-up/Polo, Zip-up, Pleated/Draped, Ruffled/Tiered, Cut-out/Open-back, Asymmetric, Graphic/Text Print, Embroidery/Applique, Sequined/Metallic, None, Unknown]
4. "fit": [Tight/Skinny/Bodycon, Slim/Tailored, Regular/Straight, Oversized/Loose/Relaxed, Flowy/Drapey, Flared/Wide-leg, Unknown]
5. "occasion": [Casual/Everyday, Office/Workwear, Party/Evening/Wedding, Sport/Active/Workout, Lounge/Sleep/Nightwear, Beach/Swimwear, Outdoor/Adventure, Intimate/Underwear, Unknown]
6. "seasonality": [Spring/Summer, Autumn/Winter, Core/All-year, Unknown]
"""

# Đã tối ưu lại Prompt để chống model nói nhảm
SYSTEM_PROMPT = f"""
You are an expert Fashion Data Annotator. Extract and infer attributes for fashion products.
If the provided text does not contain enough information to confidently guess an attribute, strictly use "Unknown".

CRITICAL RULES:
1. You MUST output ONLY a valid JSON object.
2. DO NOT wrap the JSON in markdown code blocks (no ```json).
3. DO NOT output any greetings, explanations, or conversational text.
4. You MUST use ONLY the exact values provided in the ALLOWED ENUMS.

ALLOWED ENUMS:
{ENUMS}

Output format:
{{
  "results": [
    {{
      "article_id": "...",
      "style_aesthetic": "...",
      "dominant_material": "...",
      "design_detail": "...",
      "fit": "...",
      "occasion": "...",
      "seasonality": "..."
    }}
  ]
}}
"""
print("Cell 1 Executed: Configurations loaded.")

Cell 1 Executed: Configurations loaded.


2. Helper functions

In [3]:
# [CELL 2] ĐỊNH NGHĨA CÁC HÀM XỬ LÝ (TEXT, VISION VÀ UTILS)

def get_image_base64(article_id):
    folder = article_id[:3]
    img_path = os.path.join(IMAGE_DIR, folder, f"{article_id}.jpg")
    if not os.path.exists(img_path):
        return None
    with open(img_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

async def call_api(session, payload, attempt=1):
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    try:
        async with session.post(API_URL, headers=headers, json=payload, timeout=90) as response:
            if response.status == 200:
                data = await response.json()
                content = data['choices'][0]['message']['content']
                # Dọn dẹp rác markdown nếu model lỡ sinh ra
                content = content.replace("```json", "").replace("```", "").strip()
                return json.loads(content).get("results", [])
            elif response.status == 400:
                # Lỗi 400 là lỗi cú pháp Payload, KHÔNG RETRY để tránh spam
                error_text = await response.text()
                print(f"\n[FATAL 400] Payload sai cú pháp/Tham số từ chối: {error_text}")
                return []
            else:
                error_text = await response.text()
                if attempt < 3:
                    await asyncio.sleep(2 ** attempt)
                    return await call_api(session, payload, attempt + 1)
                print(f"Error {response.status}: {error_text}")
                return []
    except Exception as e:
        if attempt < 3:
            await asyncio.sleep(2 ** attempt)
            return await call_api(session, payload, attempt + 1)
        print(f"API Failed Timeout/Network: {e}")
        return []

async def process_text_batch(session, batch_data, semaphore):
    async with semaphore:
        user_content = json.dumps(batch_data, ensure_ascii=False)
        payload = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content}
            ],
            # ĐÃ XÓA "response_format": {"type": "json_object"}
            "temperature": 0.1
        }
        return await call_api(session, payload)

async def process_vision_batch(session, batch_data, semaphore):
    async with semaphore:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        
        user_blocks = []
        text_context = []
        for item in batch_data:
            text_context.append({"article_id": item['article_id'], "raw_desc": item['raw_desc'], "refined_desc": item['refined_desc']})
            img_b64 = get_image_base64(item['article_id'])
            if img_b64:
                user_blocks.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{img_b64}", "detail": "low"}
                })
        
        user_blocks.insert(0, {"type": "text", "text": json.dumps(text_context, ensure_ascii=False)})
        messages.append({"role": "user", "content": user_blocks})
        
        payload = {
            "model": MODEL,
            "messages": messages,
            # ĐÃ XÓA "response_format": {"type": "json_object"}
            "temperature": 0.1
        }
        return await call_api(session, payload)

print("Cell 2 Executed")

Cell 2 Executed


3. Phase 1 - pipeline xử lý text 

In [ ]:
# [CELL 3] CHẠY CHẶNG 1: TEXT-ONLY PROCESSING (Saves automatically)
print(f"Loading {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV, dtype={'article_id': str})

# 1. Đọc Checkpoint để bỏ qua các ID đã xử lý (chống lỗi sập)
processed_results = {}
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            res = json.loads(line)
            processed_results[res["article_id"]] = res
    print(f"Resuming from checkpoint: {len(processed_results)} items already processed.")

# 2. Lọc ra các item chưa làm
items_to_process = []
for _, row in df.iterrows():
    aid = str(row['article_id']).zfill(10)
    if aid not in processed_results:
        items_to_process.append({
            "article_id": aid,
            "product_type": str(row.get('product_type_name', '')),
            "raw_desc": str(row.get('detail_desc', '')),
            "refined_desc": str(row.get('refined_description', ''))
        })

print(f"Remaining items to process (Text-only): {len(items_to_process)}")

# 3. Chạy Bất đồng bộ với Text
if items_to_process:
    batches = [items_to_process[i:i + BATCH_SIZE_TEXT] for i in range(0, len(items_to_process), BATCH_SIZE_TEXT)]
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    async with aiohttp.ClientSession() as session:
        tasks = [process_text_batch(session, batch, semaphore) for batch in batches]
        
        # Mở file checkpoint ở chế độ append (ghi nối)
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f_out:
            for f in tqdm(asyncio.as_completed(tasks), total=len(batches), desc="Text-Pass Relabeling"):
                batch_result = await f
                for item_res in batch_result:
                    # Ghi ngay xuống ổ cứng
                    f_out.write(json.dumps(item_res, ensure_ascii=False) + "\n")
                    processed_results[item_res["article_id"]] = item_res

print("Text Pass Complete!")

Loading ../data/processed/dataset_final_qwen_filled.csv...
Resuming from checkpoint: 1679 items already processed.
Remaining items to process (Text-only): 68518


Text-Pass Relabeling:   0%|          | 2/3426 [04:38<109:33:35, 115.19s/it]

API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 
API Failed Timeout/Network: 


Text-Pass Relabeling:   3%|▎         | 95/3426 [13:52<16:11:35, 17.50s/it] 

Error 502: <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>cloudflare</center>
</body>
</html>



...

Error 504: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>ckey.vn | 504: Gateway time-out</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />
</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl text-black-dark leading-tight

Text-Pass Relabeling:  11%|█         | 383/3426 [1:13:52<9:46:58, 11.57s/it] 

API Failed Timeout/Network: Server disconnected


CancelledError: 

API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Session is closed
API Failed Timeout/Network: Sessio

4. Phase 2 - vision fallback

In [ ]:
# [CELL 4] CHẠY CHẶNG 2: VISION FALLBACK VÀ LƯU FINAL CSV

# 1. Quét tìm các item bị "Unknown" ở các cột trọng yếu cần Vision cứu viện
items_for_vision = []
for aid, res in processed_results.items():
    # Kiểm tra xem có cột nào bị dính Unknown không
    if "Unknown" in [res.get("style_aesthetic"), res.get("dominant_material"), res.get("fit")]:
        # Lấy lại data gốc để gửi kèm
        row = df[df['article_id'] == aid].iloc[0]
        items_for_vision.append({
            "article_id": aid,
            "raw_desc": str(row.get('detail_desc', '')),
            "refined_desc": str(row.get('refined_description', ''))
        })

print(f"Items requiring Vision Fallback: {len(items_for_vision)}")

# 2. Chạy Vision API cho các ca khó
if items_for_vision:
    batches = [items_for_vision[i:i + BATCH_SIZE_VISION] for i in range(0, len(items_for_vision), BATCH_SIZE_VISION)]
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    
    async with aiohttp.ClientSession() as session:
        tasks = [process_vision_batch(session, batch, semaphore) for batch in batches]
        
        for f in tqdm(asyncio.as_completed(tasks), total=len(batches), desc="Vision-Pass Relabeling"):
            batch_result = await f
            for item_res in batch_result:
                aid = item_res.get("article_id")
                if aid in processed_results:
                    # Cập nhật kết quả mới đè lên kết quả cũ
                    processed_results[aid].update(item_res)

# 3. Tạo DataFrame cuối cùng và Merge
final_relabelled_data = list(processed_results.values())
df_new = pd.DataFrame(final_relabelled_data)

# Đổi tên các cột mới để tránh trùng (nếu CSV gốc đã có fit, occasion...)
df_new = df_new.rename(columns={
    'fit': 'fit_llm',
    'occasion': 'occasion_llm',
    'seasonality': 'seasonality_llm'
})

df_final = df.merge(df_new, on="article_id", how="left")

# Replace old columns with new LLM columns, keeping old if LLM failed completely
df_final['fit'] = df_final['fit_llm'].combine_first(df_final['fit'])
df_final['occasion'] = df_final['occasion_llm'].combine_first(df_final['occasion'])
df_final['seasonality'] = df_final['seasonality_llm'].combine_first(df_final['seasonality'])

# Xóa các cột tạm LLM sau khi đã đè
df_final = df_final.drop(columns=['fit_llm', 'occasion_llm', 'seasonality_llm'])

df_final.to_csv(OUTPUT_CSV, index=False)
print(f"Final dataset saved to: {OUTPUT_CSV}")